In [ ]:
# !pip install yfinance pandas numpy matplotlib scipy ta-lib vectorbt

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import yfinance as yf
import talib
import vectorbt as vbt
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import os, sys

In [ ]:
# ============================================================
# SCHAFF TREND CYCLE STRATEGY — Configuration
# ============================================================
TICKER = 'QQQ'
START_DATE = '2018-01-01'
TRAIN_RATIO = 0.60

INIT_CASH = 100_000
FEES = 0.0005
SLIPPAGE = 0.0005

print(f"Ticker: {TICKER}")
print(f"Start: {START_DATE} | Train ratio: {TRAIN_RATIO}")
print(f"Capital: ${INIT_CASH:,} | Fees: {FEES:.2%} | Slippage: {SLIPPAGE:.2%}")

In [ ]:
# ============================================================
# Download Data
# ============================================================
df = yf.download(TICKER, start=START_DATE, auto_adjust=True)
close = df['Close'].squeeze()

split_idx = int(len(close) * TRAIN_RATIO)
train_close = close.iloc[:split_idx]
val_close = close.iloc[split_idx:]

print(f"Total bars: {len(close)} ({close.index[0].strftime('%Y-%m-%d')} \u2192 {close.index[-1].strftime('%Y-%m-%d')})")
print(f"Train: {len(train_close)} bars ({train_close.index[0].strftime('%Y-%m-%d')} \u2192 {train_close.index[-1].strftime('%Y-%m-%d')})")
print(f"Test:  {len(val_close)} bars ({val_close.index[0].strftime('%Y-%m-%d')} \u2192 {val_close.index[-1].strftime('%Y-%m-%d')})")
print(f"\n{df.tail(5)}")

## Schaff Trend Cycle (STC) Strategy

**Indicator**: STC combines MACD momentum with Stochastic smoothing:
1. Compute MACD line = EMA(fast) - EMA(slow)
2. Apply Stochastic %K to MACD over `cycle` period → Factor1
3. Smooth Factor1 with EMA → %D1
4. Apply Stochastic %K to %D1 → Factor2
5. Smooth Factor2 with EMA → STC (0–100 oscillator)

**Entry (Long)**: STC crosses above oversold threshold (bullish momentum shift)
**Exit**: STC crosses below overbought threshold (momentum exhaustion)

**Parameters to optimize**:
- `fast_period`: Fast EMA for MACD (10–30)
- `slow_period`: Slow EMA for MACD (20–60)
- `cycle`: Stochastic cycle length (5–20)
- `oversold` / `overbought`: Signal thresholds

In [ ]:
# ============================================================
# Schaff Trend Cycle Indicator
# ============================================================
def schaff_trend_cycle(close, fast_period=23, slow_period=50, cycle=10, smooth=3):
    """
    Compute the Schaff Trend Cycle (STC).
    Returns: STC oscillator (0-100 range)
    """
    # Step 1: MACD line
    fast_ema = talib.EMA(close, timeperiod=fast_period)
    slow_ema = talib.EMA(close, timeperiod=slow_period)
    macd_line = fast_ema - slow_ema

    # Step 2: Stochastic of MACD
    macd_low = pd.Series(macd_line).rolling(cycle).min()
    macd_high = pd.Series(macd_line).rolling(cycle).max()
    stoch1 = np.where(
        (macd_high - macd_low) > 0,
        (macd_line - macd_low) / (macd_high - macd_low) * 100,
        50.0
    )

    # Step 3: Smooth with EMA
    factor1 = pd.Series(stoch1, index=close.index)
    pf1 = talib.EMA(factor1.values.astype(float), timeperiod=smooth)

    # Step 4: Second Stochastic
    pf1_series = pd.Series(pf1, index=close.index)
    pf1_low = pf1_series.rolling(cycle).min()
    pf1_high = pf1_series.rolling(cycle).max()
    stoch2 = np.where(
        (pf1_high - pf1_low) > 0,
        (pf1_series - pf1_low) / (pf1_high - pf1_low) * 100,
        50.0
    )

    # Step 5: Final smooth
    stc = pd.Series(talib.EMA(stoch2.astype(float), timeperiod=smooth), index=close.index)

    return stc.clip(0, 100)

# Test with default params
stc = schaff_trend_cycle(close)
print("STC (default params fast=23, slow=50, cycle=10):")
print(pd.DataFrame({'Close': close, 'STC': stc}).tail(10).to_string())

In [ ]:
# ============================================================
# Parameter Ranges
# ============================================================
fast_range = list(range(10, 31, 2))       # 10 to 30 step 2
slow_range = list(range(30, 61, 3))       # 30 to 60 step 3
cycle_range = list(range(5, 21, 2))       # 5 to 20 step 2
# Fixed thresholds for grid search (will test sensitivity separately)
OVERSOLD = 25
OVERBOUGHT = 75

total_combos = len(fast_range) * len(slow_range) * len(cycle_range)
print(f"Fast periods: {fast_range[0]}\u2013{fast_range[-1]} ({len(fast_range)} values)")
print(f"Slow periods: {slow_range[0]}\u2013{slow_range[-1]} ({len(slow_range)} values)")
print(f"Cycle periods: {cycle_range[0]}\u2013{cycle_range[-1]} ({len(cycle_range)} values)")
print(f"Thresholds: Oversold={OVERSOLD}, Overbought={OVERBOUGHT}")
print(f"Total combinations: {total_combos}")

In [ ]:
# ============================================================
# Grid Search (In-Sample)
# ============================================================
results = []
count = 0

for fast in fast_range:
    for slow in slow_range:
        if fast >= slow:
            continue  # fast must be < slow
        for cyc in cycle_range:
            count += 1
            try:
                stc = schaff_trend_cycle(train_close, fast, slow, cyc)

                # Entry: STC crosses above oversold (1-bar delay)
                entries_raw = (stc > OVERSOLD) & (stc.shift(1) <= OVERSOLD)
                exits_raw = (stc < OVERBOUGHT) & (stc.shift(1) >= OVERBOUGHT)
                entries = entries_raw.shift(1).fillna(False)
                exits = exits_raw.shift(1).fillna(False)

                pf = vbt.Portfolio.from_signals(
                    train_close, entries, exits,
                    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D'
                )

                n_trades = pf.trades.count()
                if n_trades < 5:
                    continue

                results.append({
                    'fast': fast,
                    'slow': slow,
                    'cycle': cyc,
                    'is_sharpe': pf.sharpe_ratio(),
                    'is_return': pf.total_return(),
                    'is_max_dd': pf.max_drawdown(),
                    'is_trades': n_trades,
                    'is_win_rate': pf.trades.win_rate(),
                    'is_profit_factor': pf.trades.profit_factor(),
                })
            except Exception:
                continue

    if count % 100 == 0:
        print(f"\ud83d\udd04 [{count}/{total_combos} combos]")

results_df = pd.DataFrame(results).sort_values('is_sharpe', ascending=False)
print(f"\n\u2705 Completed: {len(results_df)} valid configs (min 5 trades)")
print(f"\nTop 10 by Sharpe:")
print(results_df.head(10).to_string(index=False))

In [ ]:
# ============================================================
# Out-of-Sample Validation (Top 5)
# ============================================================
top5 = results_df.head(5)
oos_results = []
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)

for idx, (_, row) in enumerate(top5.iterrows()):
    fast, slow, cyc = int(row['fast']), int(row['slow']), int(row['cycle'])

    stc = schaff_trend_cycle(val_close, fast, slow, cyc)
    entries_raw = (stc > OVERSOLD) & (stc.shift(1) <= OVERSOLD)
    exits_raw = (stc < OVERBOUGHT) & (stc.shift(1) >= OVERBOUGHT)
    entries = entries_raw.shift(1).fillna(False)
    exits = exits_raw.shift(1).fillna(False)

    pf = vbt.Portfolio.from_signals(
        val_close, entries, exits,
        init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D'
    )

    oos_sharpe = pf.sharpe_ratio()
    degradation = 1 - (oos_sharpe / row['is_sharpe']) if row['is_sharpe'] > 0 else np.nan

    oos_results.append({
        'fast': fast, 'slow': slow, 'cycle': cyc,
        'is_sharpe': row['is_sharpe'],
        'oos_sharpe': oos_sharpe,
        'degradation': degradation,
        'oos_return': pf.total_return(),
        'oos_max_dd': pf.max_drawdown(),
        'oos_trades': pf.trades.count(),
    })

    eq = pf.value()
    axes[idx].plot(eq.index, eq.values, color='#4A90D9')
    axes[idx].set_title(f"F={fast} S={slow} C={cyc}\nOOS Sharpe={oos_sharpe:.2f}", fontsize=9)
    axes[idx].tick_params(labelsize=7)
    if idx == 0:
        axes[idx].set_ylabel('Portfolio Value ($)')

plt.suptitle(f'{TICKER} \u2014 Schaff Trend Cycle OOS Validation (Top 5)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

oos_df = pd.DataFrame(oos_results)
print("\nIS vs OOS Comparison:")
print(oos_df.to_string(index=False))

best = oos_df.sort_values('oos_sharpe', ascending=False).iloc[0]
BEST_FAST = int(best['fast'])
BEST_SLOW = int(best['slow'])
BEST_CYCLE = int(best['cycle'])
print(f"\n\u2705 Best OOS config: Fast={BEST_FAST}, Slow={BEST_SLOW}, Cycle={BEST_CYCLE}")
print(f"   IS Sharpe={best['is_sharpe']:.2f} \u2192 OOS Sharpe={best['oos_sharpe']:.2f} (degradation: {best['degradation']:.1%})")

In [ ]:
# ============================================================
# Parameter Sensitivity Analysis
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Fast period sensitivity
fast_sens = results_df[(results_df['slow'] == BEST_SLOW) & (results_df['cycle'] == BEST_CYCLE)].sort_values('fast')
if len(fast_sens) > 0:
    colors = ['#2ECC71' if s > 0 else '#E74C3C' for s in fast_sens['is_sharpe']]
    axes[0].bar(fast_sens['fast'], fast_sens['is_sharpe'], color=colors, alpha=0.8)
    axes[0].axvline(BEST_FAST, color='#4A90D9', linestyle='--', linewidth=2, label=f'Best={BEST_FAST}')
    axes[0].set_xlabel('Fast Period')
    axes[0].set_ylabel('IS Sharpe')
    axes[0].set_title('Fast Period Sensitivity')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

# Slow period sensitivity
slow_sens = results_df[(results_df['fast'] == BEST_FAST) & (results_df['cycle'] == BEST_CYCLE)].sort_values('slow')
if len(slow_sens) > 0:
    colors = ['#2ECC71' if s > 0 else '#E74C3C' for s in slow_sens['is_sharpe']]
    axes[1].bar(slow_sens['slow'], slow_sens['is_sharpe'], color=colors, alpha=0.8)
    axes[1].axvline(BEST_SLOW, color='#4A90D9', linestyle='--', linewidth=2, label=f'Best={BEST_SLOW}')
    axes[1].set_xlabel('Slow Period')
    axes[1].set_ylabel('IS Sharpe')
    axes[1].set_title('Slow Period Sensitivity')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

# Cycle sensitivity
cyc_sens = results_df[(results_df['fast'] == BEST_FAST) & (results_df['slow'] == BEST_SLOW)].sort_values('cycle')
if len(cyc_sens) > 0:
    colors = ['#2ECC71' if s > 0 else '#E74C3C' for s in cyc_sens['is_sharpe']]
    axes[2].bar(cyc_sens['cycle'], cyc_sens['is_sharpe'], color=colors, alpha=0.8)
    axes[2].axvline(BEST_CYCLE, color='#4A90D9', linestyle='--', linewidth=2, label=f'Best={BEST_CYCLE}')
    axes[2].set_xlabel('Cycle Period')
    axes[2].set_ylabel('IS Sharpe')
    axes[2].set_title('Cycle Period Sensitivity')
    axes[2].legend(fontsize=8)
    axes[2].grid(True, alpha=0.3)

plt.suptitle(f'{TICKER} \u2014 STC Parameter Sensitivity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Full-Sample Backtest (Best Parameters)
# ============================================================
stc = schaff_trend_cycle(close, BEST_FAST, BEST_SLOW, BEST_CYCLE)

entries_raw = (stc > OVERSOLD) & (stc.shift(1) <= OVERSOLD)
exits_raw = (stc < OVERBOUGHT) & (stc.shift(1) >= OVERBOUGHT)
entries = entries_raw.shift(1).fillna(False)
exits = exits_raw.shift(1).fillna(False)

pf = vbt.Portfolio.from_signals(
    close, entries, exits,
    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D'
)

# Metrics
total_ret = pf.total_return()
n_years = len(close) / 252
cagr = (1 + total_ret) ** (1/n_years) - 1
sharpe = pf.sharpe_ratio()
sortino = pf.sortino_ratio()
max_dd = pf.max_drawdown()
calmar = cagr / abs(max_dd) if max_dd != 0 else 0
n_trades = pf.trades.count()
win_rate = pf.trades.win_rate()
pf_ratio = pf.trades.profit_factor()

bh = vbt.Portfolio.from_holding(close, init_cash=INIT_CASH, freq='D')
bh_ret = bh.total_return()
bh_sharpe = bh.sharpe_ratio()

print(f"{'='*55}")
print(f"SCHAFF TREND CYCLE \u2014 Full Sample Results")
print(f"{'='*55}")
print(f"Params: Fast={BEST_FAST}, Slow={BEST_SLOW}, Cycle={BEST_CYCLE}")
print(f"Thresholds: Oversold={OVERSOLD}, Overbought={OVERBOUGHT}")
print(f"Period: {close.index[0].strftime('%Y-%m-%d')} \u2192 {close.index[-1].strftime('%Y-%m-%d')}")
print(f"{'\u2500'*55}")
print(f"Total Return:   {total_ret:>10.1%}  (B&H: {bh_ret:.1%})")
print(f"CAGR:           {cagr:>10.1%}")
print(f"Sharpe:         {sharpe:>10.2f}  (B&H: {bh_sharpe:.2f})")
print(f"Sortino:        {sortino:>10.2f}")
print(f"Max Drawdown:   {max_dd:>10.1%}")
print(f"Calmar:         {calmar:>10.2f}")
print(f"Trades:         {n_trades:>10d}")
print(f"Win Rate:       {win_rate:>10.1%}")
print(f"Profit Factor:  {pf_ratio:>10.2f}")

In [ ]:
# ============================================================
# Performance Dashboard
# ============================================================
fig = plt.figure(figsize=(14, 10))
gs = GridSpec(3, 1, height_ratios=[3, 1.2, 1], hspace=0.08)

STRAT_COLOR = '#4A90D9'
BENCH_COLOR = '#999999'
DD_COLOR = '#FF6B6B'

# --- Panel 1: Equity curve ---
ax1 = fig.add_subplot(gs[0])
eq = pf.value()
bh_eq = bh.value()
ax1.plot(eq.index, eq.values, color=STRAT_COLOR, linewidth=1.5,
         label=f'STC Strategy  Sharpe={sharpe:.2f}  Ret={total_ret:.1%}')
ax1.plot(bh_eq.index, bh_eq.values, color=BENCH_COLOR, linewidth=1, linestyle='--',
         label=f'Buy & Hold  Sharpe={bh_sharpe:.2f}  Ret={bh_ret:.1%}')
ax1.set_ylabel('Portfolio Value ($)', fontsize=11)
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.tick_params(labelbottom=False)
ax1.set_title(
    f'{TICKER} \u2014 Schaff Trend Cycle | F={BEST_FAST} S={BEST_SLOW} C={BEST_CYCLE}\n'
    f'CAGR: {cagr:.1%}  Sharpe: {sharpe:.2f}  MaxDD: {max_dd:.1%}  Trades: {n_trades}',
    fontsize=13, fontweight='bold'
)

# Train/test split line
split_date = close.index[split_idx]
ax1.axvline(split_date, color='orange', linestyle=':', linewidth=1, alpha=0.7, label='Train|Test split')

# --- Panel 2: STC oscillator ---
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.plot(stc.index, stc.values, color=STRAT_COLOR, linewidth=0.8)
ax2.axhline(OVERBOUGHT, color='#E74C3C', linestyle='--', linewidth=0.8, alpha=0.7)
ax2.axhline(OVERSOLD, color='#2ECC71', linestyle='--', linewidth=0.8, alpha=0.7)
ax2.fill_between(stc.index, stc.values, OVERSOLD,
                 where=stc < OVERSOLD, color='#2ECC71', alpha=0.2)
ax2.fill_between(stc.index, stc.values, OVERBOUGHT,
                 where=stc > OVERBOUGHT, color='#E74C3C', alpha=0.2)
ax2.set_ylabel('STC', fontsize=10)
ax2.set_ylim(-5, 105)
ax2.grid(True, alpha=0.3)
ax2.tick_params(labelbottom=False)

# --- Panel 3: Drawdown ---
ax3 = fig.add_subplot(gs[2], sharex=ax1)
dd = pf.drawdown() * 100
ax3.fill_between(dd.index, dd.values, 0, color=DD_COLOR, alpha=0.4)
ax3.set_ylabel('Drawdown (%)', fontsize=10)
ax3.set_xlabel('Date', fontsize=11)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Trade Analysis
# ============================================================
trades = pf.trades.records_readable

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 1. Trade returns distribution
trade_rets = trades['Return'].values * 100
winners = trade_rets[trade_rets > 0]
losers = trade_rets[trade_rets <= 0]
axes[0,0].hist(winners, bins=20, color='#2ECC71', alpha=0.7, label=f'Winners ({len(winners)})')
axes[0,0].hist(losers, bins=20, color='#E74C3C', alpha=0.7, label=f'Losers ({len(losers)})')
axes[0,0].set_xlabel('Return (%)')
axes[0,0].set_ylabel('Count')
axes[0,0].set_title('Trade Return Distribution')
axes[0,0].legend(fontsize=9)
axes[0,0].grid(True, alpha=0.3)

# 2. Cumulative P&L
cum_pnl = trades['PnL'].cumsum()
axes[0,1].plot(range(len(cum_pnl)), cum_pnl, color=STRAT_COLOR, linewidth=1.5)
axes[0,1].fill_between(range(len(cum_pnl)), cum_pnl, 0,
                        where=cum_pnl >= 0, color='#2ECC71', alpha=0.2)
axes[0,1].fill_between(range(len(cum_pnl)), cum_pnl, 0,
                        where=cum_pnl < 0, color='#E74C3C', alpha=0.2)
axes[0,1].set_xlabel('Trade #')
axes[0,1].set_ylabel('Cumulative P&L ($)')
axes[0,1].set_title('Cumulative P&L by Trade')
axes[0,1].grid(True, alpha=0.3)

# 3. Monthly returns heatmap
port_rets = pf.returns()
monthly = port_rets.resample('ME').apply(lambda x: (1+x).prod() - 1)
monthly_pivot = pd.DataFrame({
    'Year': monthly.index.year,
    'Month': monthly.index.month,
    'Return': monthly.values
}).pivot_table(values='Return', index='Year', columns='Month', aggfunc='first')
monthly_pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

im = axes[1,0].imshow(monthly_pivot.values * 100, cmap='RdYlGn', aspect='auto', vmin=-10, vmax=10)
axes[1,0].set_xticks(range(len(monthly_pivot.columns)))
axes[1,0].set_xticklabels(monthly_pivot.columns, fontsize=7)
axes[1,0].set_yticks(range(len(monthly_pivot.index)))
axes[1,0].set_yticklabels(monthly_pivot.index, fontsize=8)
for yi in range(len(monthly_pivot.index)):
    for xi in range(len(monthly_pivot.columns)):
        val = monthly_pivot.values[yi, xi]
        if not np.isnan(val):
            axes[1,0].text(xi, yi, f'{val*100:.1f}', ha='center', va='center', fontsize=7,
                          color='white' if abs(val) > 0.05 else 'black')
axes[1,0].set_title('Monthly Returns (%)')

# 4. Holding period
if 'Duration' in trades.columns:
    durations = pd.to_timedelta(trades['Duration']).dt.days
    axes[1,1].hist(durations, bins=20, color=STRAT_COLOR, alpha=0.8)
    axes[1,1].set_xlabel('Holding Period (days)')
    axes[1,1].set_ylabel('Count')
    axes[1,1].set_title(f'Holding Period (avg: {durations.mean():.0f}d)')
    axes[1,1].grid(True, alpha=0.3)
else:
    axes[1,1].text(0.5, 0.5, 'Duration data\nnot available', ha='center', va='center', fontsize=12)

plt.suptitle(f'{TICKER} \u2014 STC Trade Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Monte Carlo FTMO Simulation (10,000 paths)
# ============================================================
N_SIMS = 10_000
trade_returns = trades['Return'].values

np.random.seed(42)
sim_equity = np.ones((N_SIMS, len(trade_returns) + 1)) * INIT_CASH
for i in range(N_SIMS):
    shuffled = np.random.choice(trade_returns, size=len(trade_returns), replace=True)
    sim_equity[i, 1:] = INIT_CASH * np.cumprod(1 + shuffled)

FTMO_TARGET = INIT_CASH * 1.10
FTMO_MAX_LOSS = INIT_CASH * 0.90
hit_target = (sim_equity.max(axis=1) >= FTMO_TARGET).mean() * 100
hit_loss = (sim_equity.min(axis=1) <= FTMO_MAX_LOSS).mean() * 100
pass_rate = ((sim_equity.max(axis=1) >= FTMO_TARGET) & (sim_equity.min(axis=1) > FTMO_MAX_LOSS)).mean() * 100

if pass_rate >= 60:
    verdict = "\u2705 PASS"
elif pass_rate >= 30:
    verdict = "\u26a0\ufe0f CONDITIONAL"
else:
    verdict = "\u274c FAIL"

fig, ax = plt.subplots(figsize=(14, 5))
for pct in [5, 25, 50, 75, 95]:
    line = np.percentile(sim_equity, pct, axis=0)
    alpha = 0.3 if pct in [5, 95] else 0.5 if pct in [25, 75] else 1.0
    lw = 0.8 if pct != 50 else 2.0
    style = '--' if pct != 50 else '-'
    ax.plot(range(len(line)), line, linewidth=lw, linestyle=style, alpha=alpha,
            color='#4A90D9', label=f'P{pct}')

ax.axhline(FTMO_TARGET, color='#2ECC71', linestyle='--', linewidth=1, label='Target (+10%)')
ax.axhline(FTMO_MAX_LOSS, color='#E74C3C', linestyle='--', linewidth=1, label='Max Loss (-10%)')
ax.fill_between(range(sim_equity.shape[1]),
                np.percentile(sim_equity, 5, axis=0),
                np.percentile(sim_equity, 95, axis=0),
                color='#4A90D9', alpha=0.1)

ax.set_xlabel('Trade #', fontsize=11)
ax.set_ylabel('Equity ($)', fontsize=11)
ax.set_title(
    f'{TICKER} \u2014 Monte Carlo FTMO ({N_SIMS:,} paths)\n'
    f'Pass Rate: {pass_rate:.1f}%  |  Verdict: {verdict}',
    fontsize=13, fontweight='bold'
)
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"FTMO Monte Carlo Results:")
print(f"  Hit +10% target: {hit_target:.1f}%")
print(f"  Hit -10% limit:  {hit_loss:.1f}%")
print(f"  Clean pass rate: {pass_rate:.1f}%")
print(f"  Verdict: {verdict}")

In [ ]:
# ============================================================
# Log to Google Sheets & Export
# ============================================================
try:
    sys.path.insert(0, os.path.join(os.getcwd(), 'lib'))
    from sheets_logger import SheetsLogger

    logger = SheetsLogger()
    logger.log_result(
        ticker=TICKER,
        strategy='Schaff_Trend_Cycle',
        strategy_family='Oscillator',
        fast_param=BEST_FAST,
        slow_param=BEST_SLOW,
        filter_param=BEST_CYCLE,
        is_sharpe=float(best['is_sharpe']),
        oos_sharpe=float(best['oos_sharpe']),
        oos_return=float(best['oos_return']),
        oos_max_dd=float(best['oos_max_dd']),
        oos_trades=int(best['oos_trades']),
        full_sharpe=sharpe,
        full_return=total_ret,
        full_max_dd=max_dd,
        full_trades=n_trades,
        win_rate=win_rate,
        profit_factor=pf_ratio,
        sharpe_degradation=float(best['degradation']),
        bh_sharpe=bh_sharpe,
        bh_return=bh_ret,
        mc_ftmo_pass_rate=pass_rate,
        mc_verdict=verdict,
        data_start=START_DATE,
        data_end=close.index[-1].strftime('%Y-%m-%d'),
        total_bars=len(close),
        notes=f"Fast={BEST_FAST} Slow={BEST_SLOW} Cycle={BEST_CYCLE} OS={OVERSOLD} OB={OVERBOUGHT}"
    )
    print("\u2705 Results logged to Google Sheets")
except Exception as e:
    print(f"\u26a0\ufe0f Sheets logging skipped: {e}")

try:
    STRATEGY_NAME = "Schaff_Trend_Cycle"
    PARAM_COLS = ["fast", "slow", "cycle"]
    exec(open(os.path.join('lib', 'UNIVERSAL_EXPORT_CELL_v2.py'), encoding='utf-8').read())
except Exception as e:
    print(f"\u26a0\ufe0f Export skipped: {e}")